# A→B architecture S3/S4: does a learned Perceiver + particle model identify the transform in context?

## Description

<!-- Written before the first run (2026-10-29); kept up to date. Design: docs/design_2026-10-28_A_to_B_architecture.md §2, §8. Issue #18. Follows notebook 08 (S0 PASS, S1 GO). -->

**Question.** S1 showed that the information is there: an exhaustive Bayes learner over shifts, using only the frozen
PFN's likelihood, closes 88–95% of the A-only → oracle gap from 5 A points. Now the transform is a full monotone warp
(S3), then a warp plus an output map (S4), and enumeration no longer scales. Can a *learned* model infer it in context,
in one forward pass, from A and B?
1. Does it register at all: does it close a substantial part of the gap, compared with a non-amortised
   importance-sampling Bayes reference?
2. Can the Perceiver + A-encoder initialisation alone identify the transform (R = 0), or are the evidence blocks needed?
3. How should A be moved into B: is one particle enough (K = 1 vs 8)? Is privileged supervision needed (with vs without)?
4. Does accounting for A's evidence in B's context (re-encoding $B \cup A_{\mathrm{in}B}$ per particle) improve the
   predictive?

**Setup.** Frozen `pfn_large` (contexts $n \le 100$); $f \sim$ BNN prior; $N_B = 80$ uniform; A's design uniform;
$n_A \in \{0, 1, 2, 5, 10\}$ (nested, evaluation) / $U\{0..10\}$ (training); 32 queries.
- *S3:* $T$ = per-dimension Kumaraswamy CDF, a full-box monotone bijection, $\log\alpha, \log\beta \sim N(0, 0.5^2)$
  (no sub-windows); $h = \mathrm{id}$.
- *S4:* plus $h$ on $y$: `affine` $h(u) = c + e u$, $e \sim U(0.5, 1)$; `kum` a Kumaraswamy CDF, $\log a, \log b \sim N(0, 0.5^2)$.
  $h$ is applied to the noisy observation, so oracle densities are exact by change of variables.
- *Strata:* `identity` (sanity: the model must not lose the oracle), `warp`, `unrelated` (negative transfer; this model
  has no null particle, so harm is expected and reported).

*Model* (`experimental/a2b/model.py`):
- a Perceiver over B's frozen PFN token states and $(x, y)$, with $M = 16$ latents;
- an A-encoder;
- $K$ particles with DETR-style init;
- decoders: $T_k(x) = \sigma(\mathrm{logit}\,x + \Delta(\gamma(x), P_k))$, a residual field that is the identity at init, and,
  in S4, a monotone knot map $h_k$ (identity at init);
- $R$ weight-tied evidence blocks: likelihood, PIT, B's hidden state, a lookup into $Z_B$;
- explicit weights $\log w_k = \pi(P_k) + \sum_a \lambda_{ka}$ (pseudo-likelihood, no tempering), and a Bayes mixture as
  the output.

*Loss:* the mixture NLL at every step $r$ (deep supervision) plus a privileged term: in S3, the winner-takes-all
location error of $T_k$ at A's points; in S4, a winner-takes-all discretised KL to the oracle particle (the composite,
invariant to the T/h trade-off). 12k steps, batch 32, AdamW 3e-4, cosine. Eval: 128 fixed pairs per stratum.

| variant | change from `full` (K = 8, R = 4, supervised, predictive from B only) | answers |
|---|---|---|
| `full` | — | 1 |
| `r0` | R = 0: initialisation only | 2 |
| `k1` | K = 1 | 3 |
| `nosup` | no privileged term (NLL only) | 3 |
| `context` | last step predicts with PFN($B \cup (T_k(x_a), h_k^{-1}(y_a))$) at $T_k(x^*)$ | 4 |

*References* (same frozen PFN):
- `a_only`;
- `oracle`: B at the true $T(x^*)$, through the true $h$;
- `teacher_b`: PFN($B \cup A_{\mathrm{in}B}$) at $T(x^*)$, the ceiling for `context`;
- `is_bayes`: importance sampling over 8192 prior draws of $(T, h)$ with pseudo-likelihood weights; the S1 analogue, with
  its ESS reported.

Sweeps:
- S3, $d = 1$: all 5 variants;
- S3, $d = 2$: `full`, `r0`, `context`;
- S4, $d = 1$: $h \in$ {`affine`, `kum`} × {`full`, `nosup`}.

**What to look for.**
1. NLL vs $n_A$: the model at every step $r$ against the references.
2. Gap closure (a_only → oracle) per variant.
3. Location RMSE of the top-weight particle and the best particle (coverage) vs $n_A$ and $r$.
4. S4: the composite KL to the oracle.
5. Training curves (MLflow).
6. Examples of $T_k$ against the true $T$.

**GO / NO-GO** (*drafted by Claude, 2026-10-29, before the first run*; warp stratum, $n_A = 5$, ratio of means over pairs):
- **S3 registers (GO)** if `full` closes ≥ 50% of the a_only → oracle gap for both $d$ *and* the identity stratum stays
  within 0.1 nats/query of `oracle` at every $n_A$. **NO-GO** if it closes < 20% for either $d$. Otherwise **INCONCLUSIVE**.
- **S4 registers (GO)** if `full` closes ≥ 50% for both `affine` and `kum` ($d = 1$). Same NO-GO and INCONCLUSIVE rules.
- Descriptive, not gated:
  - Perceiver init alone: `r0`'s closure as a share of `full`'s.
  - Refinement: `full` − `r0`, paired.
  - K = 1 vs 8 at $n_A \le 2$.
  - Supervision: `nosup` vs `full`.
  - A's evidence: `context` vs `full` at $n_A = 10$, against `teacher_b`.
  - `full` vs `is_bayes`.
  - Unrelated-stratum harm.

**Limits.**
- Pilot scale: one seed, 12k steps. A NO-GO here may mean undertrained, not impossible; the training curves will show
  whether it is still improving.
- `is_bayes` with 8192 prior draws is weak where the posterior is sharp (low ESS), mostly at $d = 2$ and large $n_A$, so
  it is a reference, not a ceiling.
- No null particle: harm on unrelated pairs is expected.
- Clipping and the sigmoid parameterisation keep $T_k$ in $(0, 1)^d$.
- The privileged terms use the true $T$ and $h$; this is allowed in training on the prior only.
- Earlier reservations about warp-parameter supervision (2026-10-22) are tested by `nosup`; no parametric $\theta$ head is
  used (the decoder is a free residual field).


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/a2b/<sweep_id>"  # set after `just snapshot runs/a2b/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.